In [2]:
# ============================================================
# [SETUP] Cai ultralytics + mount Drive
# ============================================================
!nvidia-smi
!pip install ultralytics -q
import ultralytics; ultralytics.checks()

from google.colab import drive
drive.mount('/content/drive')
import os

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.3/112.6 GB disk)
Mounted at /content/drive


In [25]:
# ============================================================
# [DATA] Copy bo CanTho 70:15:15 (da chia) ve o local Colab
# ============================================================
import shutil, os
src = "/content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515"
dst = "/content/cantho_7015"
if not os.path.exists(dst):
    print("Dang copy bo CanTho...")
    shutil.copytree(src, dst)
print("Dataset san sang:", dst)
for sp in ["train","valid","test"]:
    p = os.path.join(dst, sp, "images")
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f"{sp:6s}: {n} anh")

Dataset san sang: /content/cantho_7015
train : 0 anh
valid : 166 anh
test  : 167 anh


In [26]:
# ============================================================
# [YAML] Tao data_cantho.yaml (thu tu lop khop COCO baseline)
# ============================================================
yaml_content = """train: /content/cantho_7015/train/images
val: /content/cantho_7015/valid/images
test: /content/cantho_7015/test/images

nc: 4
names: ['bus', 'car', 'motorbike', 'truck']
"""
with open("/content/data_cantho.yaml", "w") as f:
    f.write(yaml_content)
print("Da tao data_cantho.yaml")
!cat /content/data_cantho.yaml

Da tao data_cantho.yaml
train: /content/cantho_7015/train/images
val: /content/cantho_7015/valid/images
test: /content/cantho_7015/test/images

nc: 4
names: ['bus', 'car', 'motorbike', 'truck']


In [ ]:
# ============================================================
# [LOAD] Nap best.pt cua baseline YOLOv8n tu Drive
# ============================================================
from ultralytics import YOLO
import os

# best.pt cua baseline giai doan 1 (da luu tren Drive)
baseline_pt = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt"

# Neu duong dan tren khong dung, tim tu dong:
if not os.path.exists(baseline_pt):
    print("Khong thay o duong dan mac dinh, dang tim best.pt cua baseline...")
    !find "/content/drive/MyDrive/Do an AI giao thong" -name "best.pt" | grep -i baseline

print("Nap baseline tu:", baseline_pt)
model = YOLO(baseline_pt)
print("Da nap baseline YOLOv8n (mAP@0.5 goc = 0.686)")

Nap baseline tu: /content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt
Da nap baseline YOLOv8n (mAP@0.5 goc = 0.686)


In [ ]:
# ============================================================
# [FINE-TUNE] Fine-tune YOLOv8n baseline tren CanTho
# ============================================================
results = model.train(
    data="/content/data_cantho.yaml",
    epochs=100,
    patience=20,        # Early Stopping
    imgsz=640,
    batch=16,
    lr0=0.001,          # <-- LEARNING RATE THAP (chia khoa cua fine-tune)
    seed=42,
    name="finetune_cantho",
)

Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_cantho.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es

In [ ]:
# ============================================================
# [SAVE] Luu ket qua fine-tune ve Drive
# ============================================================
import shutil, os
src = "/content/runs/detect/finetune_cantho"
dst = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_cantho"
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Da luu ket qua fine-tune ve Drive:", dst)
    for f in ["weights/best.pt","results.png","results.csv","confusion_matrix.png"]:
        p = os.path.join(dst, f)
        print(f"{'OK' if os.path.exists(p) else 'X '} {f}")

Da luu ket qua fine-tune ve Drive: /content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_cantho
OK weights/best.pt
OK results.png
OK results.csv
OK confusion_matrix.png


In [ ]:
# ============================================================
# [EVAL] Danh gia best.pt (sau fine-tune) tren tap test CanTho
# ============================================================
from ultralytics import YOLO
model_ft = YOLO("/content/runs/detect/finetune_cantho/weights/best.pt")

metrics = model_ft.val(data="/content/data_cantho.yaml", split="test", name="eval_finetune_test")

print("\n===== SAU FINE-TUNE - TREN TAP TEST CANTHO =====")
print("mAP@0.5      =", round(metrics.box.map50, 4))
print("mAP@0.5:0.95 =", round(metrics.box.map, 4))
print("Precision    =", round(metrics.box.mp, 4))
print("Recall       =", round(metrics.box.mr, 4))

print("\n===== mAP@0.5 TUNG LOP =====")
names = model_ft.names
for i in range(len(metrics.box.ap_class_index)):
    c = metrics.box.ap_class_index[i]
    print(names[c], "-> mAP50 =", round(metrics.box.ap50[i], 4))

Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1701.6±489.0 MB/s, size: 41.6 KB)
val: Scanning /content/cantho_7015/test/labels... 167 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 167/167 2.1Kit/s 0.1s
val: New cache created: /content/cantho_7015/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 3.0it/s 3.7s
                   all        167        980      0.923      0.963      0.979      0.784
                   bus          6          7          1      0.995      0.995      0.931
                   car        149        494      0.957      0.984      0.992      0.806
             motorbike        155        441      0.874      0.952      0.968      0.635
                 truck         32         38      0.862      0.921      0.961 

In [ ]:
# ============================================================
# [COMPARE] Baseline goc vs Sau fine-tune (deu do tren test CanTho)
# ============================================================
from ultralytics import YOLO
import pandas as pd

# 1) Baseline goc (chua fine-tune) - danh gia trUC tiep tren test CanTho
baseline_pt = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt"
m_base = YOLO(baseline_pt)
r_base = m_base.val(data="/content/data_cantho.yaml", split="test", name="eval_baseline_on_cantho")

# 2) Sau fine-tune
m_ft = YOLO("/content/runs/detect/finetune_cantho/weights/best.pt")
r_ft = m_ft.val(data="/content/data_cantho.yaml", split="test", name="eval_ft_on_cantho")

# Bang so sanh
compare = pd.DataFrame([
    {"Trang thai":"Baseline goc (COCO)", "mAP@0.5":round(r_base.box.map50,4), "mAP@0.5:0.95":round(r_base.box.map,4), "Precision":round(r_base.box.mp,4), "Recall":round(r_base.box.mr,4)},
    {"Trang thai":"Sau fine-tune CanTho","mAP@0.5":round(r_ft.box.map50,4),   "mAP@0.5:0.95":round(r_ft.box.map,4),   "Precision":round(r_ft.box.mp,4),   "Recall":round(r_ft.box.mr,4)},
])
print("\n========== SO SANH TREN TAP TEST CANTHO ==========")
print(compare.to_string(index=False))

# Muc cai thien
d = r_ft.box.map50 - r_base.box.map50
print(f"\n=> Fine-tune cai thien mAP@0.5: {d:+.4f} ({d/r_base.box.map50*100:+.1f}%)")
print("=> Chung minh fine-tune giup model bam sat giao thong Viet Nam hon.")

Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1315.0±290.7 MB/s, size: 40.0 KB)
val: Scanning /content/cantho_7015/test/labels.cache... 167 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 167/167 31.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.9it/s 5.8s
                   all        167        980       0.71      0.614      0.647      0.455
                   bus          6          7      0.749      0.854      0.701      0.595
                   car        149        494       0.71      0.897      0.878      0.639
            motorcycle        155        441      0.743      0.125      0.354     0.0921
                 truck         32         38      0.637      0.579      0.655      0.494
Speed: 7.4ms preprocess, 4.7ms inference, 0.0

In [7]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt')

metrics = model.val(
    data='/content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515/data.yaml',
    split='test',
    imgsz=640,
    batch=16
)
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@0.5 theo tung lop:", metrics.box.maps)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.4 ms, read: 0.1±0.0 MB/s, size: 42.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515/test/labels.cache... 167 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 167/167 26.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 9.0s/it 1:39
                   all        167        980       0.71      0.614      0.647      0.455
                   bus          6          7      0.749      0.854      0.701      0.595
                   car        149        494       0.71      0.897      0.878      0.639
            motorcycle       

In [8]:
from ultralytics import YOLO

model_cantho = YOLO('/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/CanTho/weights/best.pt')

metrics_b = model_cantho.val(
    data='/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515/data.yaml',
    split='test',
    imgsz=640,
    batch=16
)

print("mAP@0.5:", metrics_b.box.map50)
print("mAP@0.5:0.95:", metrics_b.box.map)
print("Precision:", metrics_b.box.mp)
print("Recall:", metrics_b.box.mr)
print("mAP@0.5 theo tung lop:", metrics_b.box.maps)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.8±0.3 ms, read: 0.1±0.0 MB/s, size: 37.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515/test/labels.cache... 2851 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2851/2851 854.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 179/179 9.3s/it 27:49
                   all       2851      10683     0.0737     0.0714     0.0137    0.00649
                   bus        656        973     0.0524      0.117     0.0225    0.00953
                   car       1937       6901       0.19     0.0326     0.0208     0.0122
             mo

In [27]:
from ultralytics import YOLO

# Nap model CanTho huan luyen doc lap (KHONG phai model fine-tune)
model_cantho_only = YOLO('/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/CanTho/weights/best.pt')

# Danh gia tren CHINH tap test CanTho (dung file yaml da copy ve local, /content/data_cantho.yaml,
# de tranh loi doc cham tu Drive nhu da gap truoc do)
metrics_cantho_only = model_cantho_only.val(
    data="/content/data_cantho.yaml",
    split="test",
    imgsz=640,
    batch=16,
    name="eval_cantho_only_on_cantho"
)

print("===== CANTHO ONLY - TREN TAP TEST CANTHO =====")
print("mAP@0.5    =", round(metrics_cantho_only.box.map50, 4))
print("mAP@0.5:0.95 =", round(metrics_cantho_only.box.map, 4))
print("Precision  =", round(metrics_cantho_only.box.mp, 4))
print("Recall     =", round(metrics_cantho_only.box.mr, 4))

print("\n===== mAP@0.5 TUNG LOP =====")
names = model_cantho_only.names
for i in range(len(metrics_cantho_only.box.ap_class_index)):
    c = metrics_cantho_only.box.ap_class_index[i]
    print(names[c], "-> mAP50 =", round(metrics_cantho_only.box.ap50[i], 4))

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1505.4±423.8 MB/s, size: 42.4 KB)
val: Scanning /content/cantho_7015/test/labels... 167 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 167/167 2.3Kit/s 0.1s
val: New cache created: /content/cantho_7015/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 3.0it/s 3.7s
                   all        167        980      0.919       0.85      0.939      0.745
                   bus          6          7      0.906      0.714       0.89      0.808
                   car        149        494      0.979      0.943      0.988      0.812
             motorbike        155        441      0.943      0.864      0.966      0.651
                 truck         32         38      0.848      0.879      0.913 